# Lab: a first EDA of California housing

Companion notebook of the lesson *D1. Exploratory Data Analysis* — https://insper.github.io/ML/2026.2/classes/eda/

Goal: a first exploratory analysis of California Housing, done the right way: split first, explore only the training part, and end with a written list of findings. No models yet. Run the cells in order; the whole notebook takes a few seconds.

The notebook follows the two labs of the page: the **fundamental** lab first, then the **advanced** one, in the same session. Each step opens with a **Predict**: answer it before running the cell. The solutions of the simulations are on the page.

# Fundamental lab — a first EDA of California housing

## Step 1 — load and meet the columns

**Predict.** `info()` lists every column with its type and its number of non-null values. How would a column with missing values show up in that list?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.width', 120)
URL = "https://raw.githubusercontent.com/hsandmann/biblio/refs/heads/main/ml/aula02/housing.csv"
df = pd.read_csv(URL)
print(df.shape)
df.info()

## Step 2 — split first, then explore

**Predict.** `total_bedrooms` has 207 missing values. After a random 80/20 split, will exactly 80% of them land in the training part, roughly 80%, or could it be anything?

In [ ]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(df, test_size=0.2, random_state=42)
train = train.copy()          # we will add columns to train; keep test untouched
print(train.shape, test.shape)

## Step 3 — audit: missing values and duplicates

**Predict.** If the bedroom counts were missing completely at random (MCAR), how should the districts with a missing value compare with the others on income, house value and age?

In [ ]:
print(train.isna().sum()[lambda s: s > 0])
print("duplicated rows:", train.duplicated().sum())
print("more bedrooms than rooms:", (train['total_bedrooms'] > train['total_rooms']).sum())
impossible = train[train['households'] > train['population']]
print("more households than people:", len(impossible))
print(impossible[['population', 'households', 'total_rooms', 'median_house_value']])

missing = train['total_bedrooms'].isna()
print(train.groupby(missing)[['median_income', 'median_house_value', 'housing_median_age']].mean().round(2))
target = train['median_house_value']
gap = target[missing].mean() - target[~missing].mean()
print(f"difference in mean value: {gap:,.0f} · standard error for {missing.sum()} random rows: "
      f"{target.std() / np.sqrt(missing.sum()):,.0f}")

**Simulation: would the check catch a MAR mechanism?** The check above found no evidence against MCAR. Would it have found any if the values were not missing at random? On a copy of `train`, hide `total_bedrooms` in 10% of the INLAND districts (draw them with `np.random.default_rng(0)`) and repeat the comparison of the rows with and without the value. How does the gap in mean value compare with the wobble now?

In [ ]:
# your simulation here

## Step 4 — summary statistics

**Predict.** For the four counts (rooms, bedrooms, population, households), will the mean be above or below the median? And for `housing_median_age`, a column that stops at 52?

In [ ]:
num = train.select_dtypes('number')
stats = pd.DataFrame({'mean': num.mean(), 'median': num.median(),
                      'min': num.min(), 'max': num.max(), 'skew': num.skew()})
print(stats.round(2))

## Step 5 — distributions, ceilings and the log

**Predict.** In a grid of 50-bin histograms, how would a cap show up? Which columns do you expect to have one?

In [ ]:
num.hist(bins=50, figsize=(12, 8))
plt.tight_layout()
plt.show()

at_max = pd.DataFrame({'max': num.max(), 'rows at max': (num == num.max()).sum()})
at_max['share'] = (at_max['rows at max'] / len(num)).round(4)
print(at_max.sort_values('rows at max', ascending=False).head(4))

counts = ['total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']
print(pd.DataFrame({'skew': train[counts].skew(), 'skew of log': np.log(train[counts]).skew()}).round(2))

## Step 6 — the categorical column

**Predict.** Which group will have the lowest median house value? And will ISLAND, with 5 districts, end up on both sides of the split?

In [ ]:
print(train['ocean_proximity'].value_counts())
g = train.groupby('ocean_proximity')
table = pd.DataFrame({'n': g.size(), 'median value': g['median_house_value'].median(),
                      'median income': g['median_income'].median().round(2)}).sort_values('median value')
print(table)

sns.violinplot(data=train, x='ocean_proximity', y='median_house_value', order=table.index, cut=0)
plt.show()

## Step 7 — relationships with the target

**Predict.** Apart from income, which feature will correlate most with the value? Will latitude and longitude, in a state where everything depends on the coast, come out strong?

In [ ]:
features = train.select_dtypes('number').drop(columns='median_house_value')
corr = pd.DataFrame({'pearson': features.corrwith(target),
                     'spearman': features.corrwith(target, method='spearman')})
print(corr.sort_values('spearman', key=abs, ascending=False).round(3))

train.plot.hexbin(x='median_income', y='median_house_value', gridsize=40, mincnt=1)
plt.show()

**Simulation: one typo against the two coefficients.** Copy `train`, pick one district, any one, and multiply its `median_income` by 100, as if the decimal point had slipped two places. Recompute the Pearson and Spearman correlations of income with the target. Which one moves, and by how much?

In [ ]:
# your simulation here

**To close the fundamental part.** In one sentence each: (1) why can a mean and a standard deviation describe nobody in a column, and which statistics resist that? (2) Why does a correlation near zero not mean that two columns are unrelated? (3) Why must the median used to fill the missing values come from the training part only?

## Your turn (fundamental)

At home:

1. **Stratified split.** Create `income_cat = pd.cut(df['median_income'], [0, 1.5, 3, 4.5, 6, np.inf], labels=[1, 2, 3, 4, 5])` and compare the proportions of each income category in the test set for the random split of step 2 and for `train_test_split(..., stratify=income_cat)`. Which split keeps the test set closer to the whole dataset? Why does it matter most for the column that correlates best with the target?
2. **The rare category.** Repeat the split of step 2 with 1,000 different values of `random_state` and count how often the test set has no ISLAND district at all. Compare with \(0.8^5\). What would happen to a model evaluated per category?
3. **Robust or not?** On a copy of `train`, multiply the 10 largest values of `median_income` by 10, as if they had been typed in the wrong unit, and recompute the mean, the median, the standard deviation and the IQR of the column. Which of the four move, and which section of this part predicted it?

# Advanced lab — from the map to a data card

## Step 8 — the map

**Predict.** Latitude had \(r = -0.146\) with the value. Walking north one degree at a time, will the median value fall steadily, or go up and down?

In [ ]:
train.plot.scatter(x='longitude', y='latitude', c='median_house_value', cmap='viridis',
                   s=train['population'] / 100, alpha=0.4, figsize=(7, 6))
plt.show()

bands = pd.cut(train['latitude'], [32, 33, 34, 35, 36, 37, 38, 39, 40, 42])
print(train.groupby(bands, observed=True)['median_house_value'].agg(['size', 'median']))

**Simulation: give location a direction.** Latitude and longitude are not monotonic in the value, but the distance to the big coastal cities might be. For each district, compute the distance in km to the nearer of Los Angeles (34.05, −118.24) and San Francisco (37.77, −122.42) — one degree of latitude is about 111 km, and one degree of longitude 111 km times the cosine of the latitude — and correlate it with the target. Then cut it into rings and print the median value of each.

In [ ]:
# your simulation here

## Step 9 — redundant counts and ratios

**Predict.** The four counts all measure the size of the district. Will a ratio such as bedrooms per room correlate with the value more or less than the raw counts?

In [ ]:
train['rooms_per_household'] = train['total_rooms'] / train['households']
train['bedrooms_per_room'] = train['total_bedrooms'] / train['total_rooms']
train['people_per_household'] = train['population'] / train['households']

size = ['total_rooms', 'total_bedrooms', 'population', 'households']
ratios = ['rooms_per_household', 'bedrooms_per_room', 'people_per_household']
print(train[size].corr().round(2))
print(train[size + ratios].corrwith(target).round(3))

## Step 10 — outliers and odd districts

**Predict.** Will the districts the IQR rule flags be mostly errors, or mostly large, ordinary districts? And what kind of district will a ratio like people per household single out?

In [ ]:
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()

print(train[size + ['median_income', 'people_per_household']].apply(iqr_outliers))
odd = train[train['people_per_household'] > 10]
print(len(odd), "districts with more than 10 people per household")
print(odd.sort_values('people_per_household', ascending=False)
         [['population', 'households', 'people_per_household', 'median_house_value']].head(5).round(1))

**Simulation: clean every column with the IQR rule.** A common recipe is to drop every row that the IQR rule flags in any column. Apply it to the six columns of step 10 at once. How many districts would go? And what happens to the mean house value, to the share of districts at the ceiling and to the mean income of what is left?

In [ ]:
# your simulation here

## Step 11 — write down the findings

**Predict.** Of everything found so far, which finding will matter most for the model that will be trained on this data?

In [ ]:
findings = {
    'training rows': len(train),
    'impossible rows (more households than people)': len(impossible),
    'missing total_bedrooms (no evidence against MCAR)': train['total_bedrooms'].isna().sum(),
    'target at the 500,001 ceiling': (target >= 500001).sum(),
    'housing_median_age at the 52 ceiling': (train['housing_median_age'] >= 52).sum(),
    'ISLAND districts': (train['ocean_proximity'] == 'ISLAND').sum(),
    'districts with > 10 people per household': len(odd),
    'strongest feature (Spearman)': corr['spearman'].abs().idxmax(),
}
for k, v in findings.items():
    print(f"{k:50s} {v}")

### Your findings

Write them here, one sentence per finding.

**Facts about the data**

- 

**Problems to fix in preprocessing**

- 

**Hypotheses to test with models**

- 

**To close the lesson.** In one sentence each: (1) why does `bedrooms_per_room` correlate with the value more than `total_bedrooms`, and why is that not yet a finding? (2) Why are most of the districts flagged by the IQR rule not errors? (3) Why is a district recorded at 500,001 not worth 500,001?

## Your turn (advanced)

At home, with more time:

1. **Find the caps automatically.** Write a function that returns every numeric column whose maximum *or* minimum is shared by more than 1% of the rows. Which columns does it find? Which cap does it miss at a 1% threshold, and what does that say about choosing thresholds?
2. **Pearson vs Spearman.** For `rooms_per_household`, compute both coefficients with the target, then recompute them after removing the districts with more than 10 people per household. Which coefficient moves more, and why?
3. **Separate location from income.** Within each income quintile of `train` (`pd.qcut(train['median_income'], 5)`), compare the median value of the INLAND districts with that of all the others. Does the gap of step 6 survive once income is fixed? Is it larger or smaller than the overall gap?
4. **Who sits at the ceiling?** For each income quintile, compute the share of districts at the 500,001 ceiling. In which quintile will a model trained on this target underestimate the most, and why?